# Notebook 10 — Mixture-of-Experts Transformers from First Principles

    ## Learning objectives

    - Derive sparse MoE routing, expert aggregation, and parameter-versus-compute scaling
- Implement top-k routing with capacity and auxiliary load-balancing losses
- Explain expert parallelism, communication costs, collapse, and production diagnostics

    Cells labeled **optional GPU/remote** are deliberately guarded. Read them first,
    then opt in when the required hardware or Hugging Face Inference access is available.


In [ ]:
# Colab/local environment setup — run this cell first.
import importlib.util
import os
import platform
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
PACKAGES = []

if IN_COLAB and PACKAGES:
    print("Installing notebook dependencies in the Colab runtime...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *PACKAGES])

# Load an HF token from Colab Secrets without displaying it. In Colab, create a
# secret named HF_TOKEN (or HUGGINGFACE_TOKEN) and enable notebook access.
if IN_COLAB:
    from google.colab import userdata
    token = None
    for secret_name in ("HF_TOKEN", "HUGGINGFACE_TOKEN"):
        try:
            token = userdata.get(secret_name)
        except Exception:
            pass
        if token:
            break
    if token:
        os.environ["HF_TOKEN"] = token
        os.environ["HUGGINGFACE_TOKEN"] = token
else:
    try:
        from dotenv import load_dotenv
        load_dotenv(".env")
    except ImportError:
        pass

try:
    import torch
    accelerator = torch.cuda.get_device_name(0) if torch.cuda.is_available() else (
        "Apple MPS" if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available() else "CPU"
    )
    print(f"runtime={platform.platform()} | Python={platform.python_version()} | accelerator={accelerator}")
    if False and not torch.cuda.is_available():
        print("WARNING: this training notebook is designed for a Colab GPU runtime. "
              "Select Runtime > Change runtime type > T4 GPU (or better).")
except ImportError:
    print(f"runtime={platform.platform()} | Python={platform.python_version()}")

print("Hugging Face token configured:", bool(os.getenv("HUGGINGFACE_TOKEN")))


## 13.1 Sparse capacity

A dense feed-forward layer applies the same parameters to every token. A sparse
mixture-of-experts (MoE) layer owns several feed-forward networks but routes each token to only
a small subset. If a transformer has (E) experts and activates (k\ll E), total parameters can
grow much faster than floating-point work per token. This is conditional computation—not an
ensemble of separately decoded models. Attention is commonly dense while selected MLP sublayers
become experts.

For hidden state (x_t), a router produces logits (r_t=W_rx_t), probabilities
(p_t=softmax(r_t)), and a top-k set (S_t). The output is
(y_t=\sum_{e\in S_t}\tilde p_{t,e}E_e(x_t)), with selected weights often renormalized.
Routing is token-level, so tokens from one sequence may visit different experts. Total parameters,
active parameters, FLOPs, memory, and communication must therefore be reported separately.


In [ ]:
import torch
from torch import nn
torch.manual_seed(7)
tokens, width, experts, top_k = 12, 16, 4, 2
hidden = torch.randn(tokens, width)
router = nn.Linear(width, experts, bias=False)
probabilities = router(hidden).softmax(-1)
weights, indices = probabilities.topk(top_k, dim=-1)
weights = weights / weights.sum(-1, keepdim=True)
print("routes:", indices[:5].tolist())
print("selected weights sum:", weights.sum(-1)[:5])


## 13.2 Dispatch, combine, and capacity

An implementation groups tokens by selected expert, runs expert matrix multiplications, weights
the results, and scatters them back to original order. A naïve Python loop is readable; optimized
grouped GEMM kernels avoid many tiny operations. In distributed systems, experts reside on
different devices and all-to-all communication dispatches token representations. Poor routing can
leave some accelerators idle while others overflow.

Training systems commonly give each expert finite capacity, approximately
`capacity_factor × tokens × k / experts`. Overflow tokens may be dropped, rerouted, or handled by
a shared expert. Dropping changes the computation and can damage important or minority tokens;
very high capacity wastes memory and padding. Capacity, batch composition, sequence packing, and
data-parallel topology interact, so routing must be measured on realistic batches.


In [ ]:
class Expert(nn.Module):
    def __init__(self, d):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d, 4*d), nn.SiLU(), nn.Linear(4*d, d))
    def forward(self, x): return self.net(x)

bank = nn.ModuleList([Expert(width) for _ in range(experts)])
combined = torch.zeros_like(hidden)
for expert_id, expert in enumerate(bank):
    token_pos, slot = torch.where(indices == expert_id)
    if len(token_pos):
        combined.index_add_(0, token_pos, expert(hidden[token_pos]) * weights[token_pos, slot, None])
print(combined.shape, "finite:", torch.isfinite(combined).all().item())


## 13.3 Why routing needs regularization

Task loss alone can collapse traffic onto a few initially favored experts. A load-balancing
objective encourages agreement between the fraction of tokens assigned to each expert and mean
router probability. Router z-loss penalizes large log-sum-exp values and improves numerical
stability. Noise or jitter during training can encourage exploration. These terms are not free:
overly strong balance prevents useful specialization, while global balance can conceal imbalance
within languages, domains, positions, or devices.

Router gradients through hard top-k selection are subtle. Selected routing weights remain
differentiable, but discrete membership is not; practical formulations use soft probabilities in
auxiliary objectives. Track router entropy, top-1 and top-k shares, overflow/drop rate, expert
utilization, per-expert gradient/update norms, and routing by meaningful data slice.


In [ ]:
top1 = probabilities.argmax(-1)
assignment_fraction = torch.bincount(top1, minlength=experts).float() / tokens
mean_probability = probabilities.mean(0)
balance_loss = experts * (assignment_fraction * mean_probability).sum()
z_loss = torch.logsumexp(router(hidden), dim=-1).square().mean()
entropy = -(probabilities * probabilities.clamp_min(1e-9).log()).sum(-1).mean()
print({"assignment": assignment_fraction.tolist(), "balance": balance_loss.item(),
       "z_loss": z_loss.item(), "entropy": entropy.item()})


## 13.4 Expert parallelism and inference

Data parallelism replicates experts; expert parallelism shards them. Tensor, pipeline, and expert
parallelism can be combined, but every axis introduces placement and collective-communication
constraints. All-to-all volume, network topology, token skew, grouped-matrix efficiency, and
overlap of communication with compute determine realized speed. A model with low active FLOPs can
still be slow if dispatch dominates or each expert receives too few tokens.

Decode batches are smaller and more dynamic than training batches, which can reduce expert-kernel
efficiency. Expert weights may exceed one device even though active weights per token are small.
Quantization, caching, speculative decoding, and batching have architecture-specific support.
Serving claims should include concurrency and routing distribution, not only single-request latency.


In [ ]:
def moe_accounting(d_model, d_ff, num_experts, active_experts):
    per_expert = 2 * d_model * d_ff
    return {"expert_parameters": per_expert * num_experts,
            "active_expert_parameters_per_token": per_expert * active_experts,
            "active_fraction": active_experts / num_experts}
for e, k in [(8, 2), (64, 2), (128, 4)]: print(e, k, moe_accounting(4096, 14336, e, k))


## 13.5 Evaluation and design choices

Compare an MoE model with a dense baseline under matched training tokens and either matched active
compute or matched wall-clock budget. Evaluate quality, throughput, memory, communication, and
stability. Inspect expert specialization cautiously: frequent routing correlation does not prove a
human-interpretable expert function. Ablate experts and routing only with distribution-aware tests.

Top-1 routing is cheaper but gives fewer paths; top-2 can improve robustness at extra compute and
communication. Shared experts provide always-on capacity. Fine-grained experts change kernel shapes
and routing granularity. Device-limited inference may favor smaller dense models despite MoE quality.
The correct architecture depends on training fabric and target serving topology, not parameter count
marketing. Preserve router configuration, capacity rules, expert mapping, auxiliary coefficients,
and backend versions in checkpoints and model cards.


## 10.5 Routing capacity and auxiliary objectives

Top-k routing converts gate logits into expert assignments and weights. Real systems have finite expert capacity; overflow tokens may be dropped, rerouted, or padded, each affecting quality and efficiency. Load-balancing losses encourage more even traffic, while router z-loss can discourage extreme logits. These auxiliary terms need explicit coefficients and logging because they can compete with language-model loss. Report tokens per expert, capacity utilization, overflow, entropy, and routing by domain. Balanced counts alone do not prove experts learned useful specialization, and perfectly uniform routing can defeat conditional computation.


In [ ]:
torch.manual_seed(3); router=torch.randn(24,4); probs=router.softmax(-1); expert=probs.argmax(-1); counts=torch.bincount(expert,minlength=4); fraction=counts/counts.sum(); mean_prob=probs.mean(0); balance=4*(fraction*mean_prob).sum()
print("counts",counts.tolist(),"load-balance proxy",balance.item(),"entropy",(-(probs*probs.log()).sum(-1).mean()).item())


## 10.6 Expert parallel communication

Experts are often partitioned across devices. Routing therefore requires an all-to-all exchange that sends token representations to owning devices and returns expert outputs. Communication volume, load imbalance, small matrix multiplications, padding to capacity, and synchronization can erase theoretical FLOP savings. Distinguish total parameters from active parameters and from bytes moved per token. Benchmark end-to-end tokens per second at realistic batch and sequence distributions rather than extrapolating from one expert matrix multiply. Checkpoint layout, expert placement, failure recovery, and serving support are architectural constraints, not later implementation details.


In [ ]:
tokens,width,experts,top_k,bytes_per=4096,4096,8,2,2
logical_routed_bytes=tokens*width*top_k*bytes_per
print("one-way routed activation MiB",logical_routed_bytes/2**20,"round-trip MiB",2*logical_routed_bytes/2**20)
for imbalance in (1.0,1.25,1.5): print("effective padded traffic MiB",2*logical_routed_bytes*imbalance/2**20)


## Exercises

    1. Turn the dispatch demonstration into a batched MoE module and verify gradients reach selected experts.
2. Sweep router temperature and plot entropy, imbalance, and task loss.
3. Design an expert-parallel placement for two nodes and identify every all-to-all boundary.

    ## Checkpoint

    Explain the notebook's central mechanism without using library names, then identify
    one assumption you would test before applying it to a real workload.
